# Exercício 1 — Python + MySQL + DataFrame

**Objetivo:** conectar no MySQL, ler a tabela `ALUNO`, transformar em DataFrame Pandas, exibir os dados e calcular a média das notas.

```sql
CREATE TABLE ALUNO (
    idAluno INT PRIMARY KEY AUTO_INCREMENT,
    nome VARCHAR(50),
    curso VARCHAR(30),
    nota DECIMAL(4,2)
);
```

## 1. Importações

In [ ]:
import os
from pathlib import Path

import pandas as pd
import mysql.connector
from dotenv import load_dotenv

try:
    from IPython.display import display
except ImportError:
    display = print

EXERCICIO_DIR = Path.cwd()
if not (EXERCICIO_DIR / ".env").exists():
    candidato = Path(__file__).resolve().parent if "__file__" in dir() else EXERCICIO_DIR

    for pasta in [EXERCICIO_DIR, EXERCICIO_DIR / "exercicios" / "python_mysql_dataframe"]:
        if (pasta / ".env").exists() or (pasta / ".env.example").exists():
            EXERCICIO_DIR = pasta
            break
load_dotenv(EXERCICIO_DIR / ".env")

print("Bibliotecas carregadas.")


Bibliotecas carregadas.


## 2. Conexão com o banco MySQL

In [2]:
config = {
    "host": os.getenv("MYSQL_HOST", "127.0.0.1"),
    "port": int(os.getenv("MYSQL_PORT", "3307")),
    "user": os.getenv("MYSQL_USER", "aluno"),
    "password": os.getenv("MYSQL_PASSWORD", "aluno123"),
    "database": os.getenv("MYSQL_DATABASE", "escola"),
}

conexao = mysql.connector.connect(**config)

print("Conexão estabelecida com sucesso!")
print(f"Banco: {config['database']} | Host: {config['host']}:{config['port']}")

Conexão estabelecida com sucesso!
Banco: escola | Host: 127.0.0.1:3307


## 3. Executar `SELECT * FROM ALUNO`

In [3]:
cursor = conexao.cursor()
cursor.execute("SELECT * FROM ALUNO")

registros = cursor.fetchall()       # lista de tuplas com os dados
colunas = [desc[0] for desc in cursor.description]  # nomes das colunas

print(f"Total de registros retornados: {len(registros)}")
print("Colunas:", colunas)
print("\nPrimeiros registros (brutos):")
for linha in registros[:3]:
    print(linha)

Total de registros retornados: 8
Colunas: ['idAluno', 'nome', 'curso', 'nota']

Primeiros registros (brutos):
(1, 'Ana Souza', 'ADS', Decimal('8.50'))
(2, 'Bruno Lima', 'ADS', Decimal('7.00'))
(3, 'Carla Mendes', 'SI', Decimal('9.25'))


## 4. Transformar o resultado em DataFrame Pandas

In [4]:
df_aluno = pd.DataFrame(registros, columns=colunas)

# Garante que a nota seja numérica para o cálculo da média
df_aluno["nota"] = pd.to_numeric(df_aluno["nota"], errors="coerce")

print("DataFrame criado!")
print("Shape:", df_aluno.shape)
print("Tipos:\n", df_aluno.dtypes)

DataFrame criado!
Shape: (8, 4)
Tipos:
 idAluno      int64
nome        object
curso       object
nota       float64
dtype: object


## 5. Exibir os dados

In [ ]:
print("=== Tabela ALUNO ===")
display(df_aluno)

## 6. Calcular a média das notas com Pandas

In [ ]:
media_notas = df_aluno["nota"].mean()

print(f"Média das notas: {media_notas:.2f}")


print("\nMédia por curso:")
display(df_aluno.groupby("curso")["nota"].mean().round(2).reset_index(name="media"))

## 7. Fechar conexão

In [ ]:
cursor.close()
conexao.close()
print("Conexão encerrada.")

## Forma alternativa (mais curta)

O Pandas também consegue ler direto do MySQL com `pd.read_sql`:

In [ ]:

conexao = mysql.connector.connect(**config)

df_rapido = pd.read_sql("SELECT * FROM ALUNO", conexao)
print("Média (forma rápida):", round(df_rapido["nota"].mean(), 2))
display(df_rapido)

conexao.close()